# M1 — Road Damage Detection: Data Preprocessing

## Project Overview

This notebook documents the M1 preprocessing workflow for
the Road Damage Detection project.

### Workflow
1. Dataset preparation and organisation
2. Image and annotation auditing
3. Duplicate detection and review
4. Train/validation/test leakage checks
5. Classification crop generation
6. Crop verification
7. Data handover to M2

### Implementation
The preprocessing logic is maintained in Python scripts
inside the `scripts/` directory.

This notebook provides explanations, verification results,
and visualisations without unnecessarily repeating preprocessing.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project directory:", PROJECT_ROOT)
print("Scripts folder exists:", (PROJECT_ROOT / "scripts").exists())
print("Processed data exists:", (PROJECT_ROOT / "data" / "processed").exists())


## Step 1 — Loading and Inspecting the M1 Dataset

### Objective
Load and inspect the processed dataset manifests generated during M1 preprocessing without repeating the original processing steps.

### Input Files
Three CSV files are loaded from `data/processed/`:

1. **split_manifest.csv** — Contains 38,384 records describing the dataset splits.
2. **annotations.csv** — Contains 55,002 annotation records associated with road-damage images.
3. **classification_manifest.csv** — Contains 32,692 generated classification crop records.

### Classification Manifest Structure
The classification manifest includes four columns:

- `crop_path`: Location of the generated crop image.
- `image_id`: Identifier of the original source image.
- `split`: Dataset partition (train, validation, or test).
- `class`: Road-damage category (e.g., D00).

### Methodology
The Python code uses the Pandas library to:

1. Read the three existing CSV files.
2. Count the total records in each dataset.
3. Identify the classification manifest columns.
4. Display the first five classification crop records.

### Expected Output
- Total dataset record counts.
- Classification manifest column names.
- Preview of the classification dataset.

### Relevance to M2
The classification manifest provides the crop paths, class labels, and dataset partitions required for subsequent model training and evaluation.

**Note:** This step only inspects existing M1 outputs. It does not regenerate crops, modify dataset splits, or verify data leakage.


In [ ]:

import pandas as pd

# Load existing M1 outputs
data_dir = PROJECT_ROOT / "data" / "processed"

split_df = pd.read_csv(data_dir / "split_manifest.csv")
annotations_df = pd.read_csv(data_dir / "annotations.csv")
classification_df = pd.read_csv(data_dir / "classification_manifest.csv")

print("M1 DATASET SUMMARY")
print("-------------------------")
print("Split manifest records:", len(split_df))
print("Annotation records:", len(annotations_df))
print("Classification crop records:", len(classification_df))

print("\nClassification manifest columns:")
print(classification_df.columns.tolist())

display(classification_df.head())



## Step 2 — Loading and Inspecting the M1 Dataset

### Objective
Load and inspect the processed dataset manifests generated during M1 preprocessing without repeating the original processing steps.

### Input Files
Three CSV files are loaded from `data/processed/`:

1. **split_manifest.csv** — Contains 38,384 records describing the dataset splits.
2. **annotations.csv** — Contains 55,002 annotation records associated with road-damage images.
3. **classification_manifest.csv** — Contains 32,692 generated classification crop records.

### Classification Manifest Structure
The classification manifest includes four columns:

- `crop_path`: Location of the generated crop image.
- `image_id`: Identifier of the original source image.
- `split`: Dataset partition (train, validation, or test).
- `class`: Road-damage category (e.g., D00).

### Methodology
The Python code uses the Pandas library to:

1. Read the three existing CSV files.
2. Count the total records in each dataset.
3. Identify the classification manifest columns.
4. Display the first five classification crop records.

### Expected Output
- Total dataset record counts.
- Classification manifest column names.
- Preview of the classification dataset.

### Relevance to M2
The classification manifest provides the crop paths, class labels, and dataset partitions required for subsequent model training and evaluation.

**Note:** This step only inspects existing M1 outputs. It does not regenerate crops, modify dataset splits, or verify data leakage.


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# Count crops by dataset split
split_counts = classification_df["split"].value_counts()

print("CLASSIFICATION CROPS BY SPLIT")
print(split_counts)

# Count crops by damage class
class_counts = classification_df["class"].value_counts()

print("\nCLASSIFICATION CROPS BY CLASS")
print(class_counts)

# Show distribution across splits and classes
distribution = pd.crosstab(
    classification_df["class"],
    classification_df["split"]
)

display(distribution)

# Visualise train/validation/test distribution
split_counts.plot(kind="bar", figsize=(7, 4))
plt.title("Classification Crops by Dataset Split")
plt.xlabel("Dataset Split")
plt.ylabel("Number of Crops")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Step 3 Country-Based Dataset Splitting and Domain Shift

To evaluate cross-country generalisation and minimise data leakage, the dataset was partitioned by geographical source rather than randomly splitting individual images.

- **Training:** India, Japan, and United States (23,016 images).
- **Validation:** China Drone, China MotorBike, and Czech (7,207 images).
- **Testing:** Norway (8,161 images).

Norway was completely excluded from training and validation, providing an independent country-level holdout for evaluating domain shift.

The seven data sources represent six countries because China contributes two separate image sources.

This splitting strategy allows the classification and detection models to be evaluated on geographically distinct images. The held-out test set must remain excluded from model training, hyperparameter tuning, and model selection.

In [ ]:
# Identify the source/country column
source_col = next(
    (c for c in ["country", "source", "dataset"]
     if c in df.columns),
    None
)

if source_col is None:
    raise ValueError("Source column not found. Check columns above.")

# Count unique original images per source and split
counts = (
    df.groupby([source_col, "split"])["image_id"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=["train", "val", "test"], fill_value=0)
)

counts["Total"] = counts.sum(axis=1)

print("ORIGINAL IMAGE COUNT PER SOURCE AND SPLIT")
display(counts)

print("\nTOTAL IMAGES PER SPLIT")
display(counts[["train", "val", "test"]].sum())

print("\nGRAND TOTAL:", counts["Total"].sum())

# Visualization
ax = counts[["train", "val", "test"]].plot(
    kind="bar",
    figsize=(12, 6),
    width=0.8
)

ax.set_title("Original Image Count by Source and Dataset Split")
ax.set_xlabel("Source / Country")
ax.set_ylabel("Number of Original Images")
ax.legend(title="Split")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

###  Step 4 Class Distribution and Dataset Balance Assessment

**Objective:** Evaluate the distribution of road-damage classes (D00, D10, D20, and D40) across the training, validation, and test datasets.

**Methodology:**

- Calculate the number of classification crops belonging to each damage class within each dataset split.
- Calculate the percentage contribution of each class to its respective dataset split.
- Compare class distributions to identify potential class imbalance.

**Interpretation:**

The training dataset contains 3,000 crops per class, resulting in a balanced training distribution of 12,000 crops.

The validation and test datasets retain their naturally occurring class distributions and exhibit class imbalance, particularly for D20 and D40.

**Implications for model development:**

The balanced training dataset supports learning across all four classes, while the validation and test datasets allow performance evaluation under different class distributions.

For M2 and M3, model performance should be assessed using macro-F1, per-class precision and recall, and confusion matrices rather than relying solely on overall accuracy.

In [ ]:

# Classification crops by class and split
class_split = pd.crosstab(
    classification_df["class"],
    classification_df["split"]
)

display(class_split)

# Percentage distribution within each split
class_percent = (
    class_split.div(class_split.sum(axis=0), axis=1) * 100
).round(2)

print("Class distribution (%) within each split:")
display(class_percent)


###  step 5 Verification of Norway Test Dataset Annotation Availability

**Objective:** Verify that all images in the geographically held-out Norway test dataset have corresponding XML annotation files.

**Methodology:**
1. Filter the dataset manifest to select Norway test images.
2. Check whether each corresponding XML annotation file exists.
3. Calculate the total number of available and missing annotation files.
4. Display sample annotation paths for verification.

**Results:**
- Total Norway test images: 8,161
- Available XML annotation files: 8,161
- Missing XML annotation files: 0
- Annotation file availability: 100%

**Conclusion:** All 8,161 Norway test images have corresponding XML annotation files available, supporting the use of this labelled dataset for independent evaluation of model performance and domain shift.

In [ ]:
from pathlib import Path
import pandas as pd

norway_test = split_df[
    (split_df["country"] == "Norway-002") &
    (split_df["split"] == "test")
].copy()    

norway_test["xml_exists"] = norway_test["xml_path"].apply(
    lambda p: Path(str(p)).is_file() if pd.notna(p) else False
)

print("Norway test images:", len(norway_test))
print("XML files found:", norway_test["xml_exists"].sum())
print("XML files missing:", (~norway_test["xml_exists"]).sum())

print("\nExample annotation paths:")
for path in norway_test["xml_path"].head(3):
    print(repr(path))

### step 6 XML Annotation Integrity and Class Distribution Verification

**Objective:** Validate the readability of XML annotation files in the held-out Norway test dataset and examine the distribution of road-damage classes.

**Methodology:**
1. Parse all Norway test XML files using Python's ElementTree library.
2. Identify readable and unreadable XML files.
3. Extract object-level damage labels from each XML file.
4. Count the annotations belonging to each road-damage category.

**Results:**
- Total readable XML files: 8,161
- Unreadable XML files: 0
- XML parsing success rate: 100%
- Total damage annotations: 11,229

**Class Distribution:**
- D00 (Longitudinal cracks): 8,570
- D10 (Transverse cracks): 1,730
- D20 (Alligator cracks): 468
- D40 (Potholes): 461

**Interpretation:** All XML files were successfully parsed, confirming their structural readability. However, the annotation distribution is imbalanced, with longitudinal cracks substantially more frequent than other damage categories.

**Conclusion:** The Norway test dataset has readable XML annotations for all images and can support evaluation of classification performance across four damage classes. Macro-averaged F1-score, per-class precision, recall, and confusion matrices should be reported to account for class imbalance.

In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter

label_counts = Counter()
valid_xml = 0
invalid_xml = 0

for path in norway_test["xml_path"]:
    try:
        root = ET.parse(path).getroot()
        valid_xml += 1

        for obj in root.findall(".//object"):
            name = obj.findtext("name")
            if name:
                label_counts[name] += 1

    except (ET.ParseError, OSError):
        invalid_xml += 1

print("Readable XML files:", valid_xml)
print("Unreadable XML files:", invalid_xml)
print("Damage annotations:", dict(label_counts))

### step 7 Cross-Split Duplicate Candidate Group Analysis

**Objective:** Assess the distribution of potential duplicate-image groups across training, validation, and test splits to identify possible data leakage.

**Methodology:**
1. Group candidate images using their assigned `candidate_group` identifiers.
2. Calculate the number of images within each candidate group.
3. Determine how many distinct dataset splits each group spans.
4. Identify groups containing images from multiple dataset splits.
5. Display the first ten candidate groups for inspection.

**Results:**
- Total candidate groups: 388
- Candidate groups crossing dataset splits: 388
- Candidate groups confined to one split: 0

**Interpretation:** All identified candidate groups contain images from at least two dataset splits. However, similarity-based grouping does not establish that the images are genuine duplicates. Similar road surfaces, textures, and damage patterns may produce false-positive duplicate candidates.

**Conclusion:** The analysis identifies 388 cross-split candidate groups requiring consideration in the data leakage assessment. Manual verification and exact-duplicate checks are necessary to distinguish actual duplication from visual similarity.

In [ ]:
import pandas as pd
from pathlib import Path

# Find the existing cross-split candidate review file
candidates = [
    Path("../reports/m1/cross_split_candidates.csv"),
    Path("reports/m1/cross_split_candidates.csv")
]

review_path = next(
    (p for p in candidates if p.exists()), None
)

if review_path is None:
    raise FileNotFoundError(
        "cross_split_candidates.csv not found"
    )

review_df = pd.read_csv(review_path)

print("Loaded:", review_path)
print("Columns:", review_df.columns.tolist())
print("Rows:", len(review_df))

In [ ]:
# Identify candidate groups spanning multiple splits
group_summary = review_df.groupby("candidate_group").agg(
    image_count=("image_id", "nunique"),
    split_count=("split", "nunique"),
    splits=("split", lambda x: ", ".join(sorted(x.unique())))
)

cross_split_groups = group_summary[
    group_summary["split_count"] > 1
]

print("Total candidate groups:",
      len(group_summary))

print("Groups crossing dataset splits:",
      len(cross_split_groups))

print("Groups within one split:",
      len(group_summary) - len(cross_split_groups))

display(cross_split_groups.head(10))

### step 8 Manual Verification of Potential Cross-Split Duplicates

**Objective:** Validate potential duplicate-image pairs identified during the data leakage assessment.

**Methodology:**
- Load the manual review decisions from `ranked_pair_review_decisions.csv`.
- Review 50 selected image pairs, consisting of 30 highest-ranked similarity candidates and 20 randomly selected remaining candidates.
- Record the classification status and supporting observations for each reviewed pair.
- Inspect the review records to document potential cross-split duplication.

**Results:**
- Total reviewed pairs: 50
- Highest-ranked candidates: 30
- Randomly selected candidates: 20
- Pairs classified as different scenes: 50
- Confirmed duplicates among reviewed pairs: 0

**Interpretation:** All 50 reviewed image pairs were classified as different road scenes based on visual characteristics, including road layout, markings, and surrounding environments.

**Conclusion:** No confirmed duplicates were identified among the 50 manually reviewed pairs. These findings support the initial leakage assessment, although they do not exclude possible duplicates among unreviewed candidates.

In [ ]:
import pandas as pd

decision_path = (
    PROJECT_ROOT
    / "reports"
    / "m1"
    / "ranked_pair_review_decisions.csv"
)

decisions = pd.read_csv(decision_path)

print("Total reviewed records:", len(decisions))
print("\nColumns:")
print(decisions.columns.tolist())

display(decisions.head(10))

### step 9 Cross-Split Data Leakage Assessment

A duplicate-screening procedure was performed to identify potentially overlapping images across the training, validation, and test datasets.

**Results:**
- 388 candidate groups were flagged for potential cross-split similarity.
- 50 image pairs were manually reviewed.
- 30 pairs were selected from the highest-ranked similarity candidates.
- 20 additional pairs were randomly selected.
- All 50 reviewed pairs were classified as different road scenes.
- No confirmed duplicates were identified among the reviewed pairs.

**Conclusion:**

The manual review did not identify confirmed data leakage among the inspected pairs. However, unreviewed similarity candidates remain, so the absence of leakage across the entire dataset cannot yet be guaranteed.

The country-level split was maintained to support domain-shift evaluation.

In [ ]:
print("DUPLICATE REVIEW SUMMARY")
print("-" * 40)

print("Total reviewed pairs:", len(decisions))

print("\nDecision status:")
print(decisions["status"].value_counts(dropna=False))

print("\nSelection method:")
print(decisions["selection"].value_counts(dropna=False))

print("\nUnreviewed or uncertain decisions:")
uncertain = decisions[
    decisions["status"].isna()
    | decisions["status"].astype(str).str.contains(
        "review|uncertain|pending",
        case=False,
        regex=True
    )
]

display(uncertain)

### step 10 Classification Crop Dataset — Metadata Validation

**Objective:** Verify the completeness, class distribution, and dataset split distribution of the generated classification crops.

**Results:**
- Total classification crops: **32,692**
- Training crops: 12,000
- Validation crops: 9,463
- Test crops: 11,229
- Missing metadata values: 0
- Damage classes: D00, D10, D20, D40

**Observations:**
- All crop records contain the required metadata fields: `crop_path`, `image_id`, `split`, and `class`.
- D00 is the most represented class, indicating class imbalance.
- The dataset contains all three required splits.

**Conclusion:** The classification crop metadata is complete. Further verification is required to confirm that all crop images are readable and that their labels and splits match the original annotations and source images.

In [ ]:
from pathlib import Path
from PIL import Image
import pandas as pd

print("Available columns:")
print(classification_df.columns.tolist())

print("\nTotal classification records:")
print(len(classification_df))

print("\nMissing values:")
print(classification_df.isna().sum())

print("\nClass distribution:")
print(classification_df["class"].value_counts())

print("\nSplit distribution:")
print(classification_df["split"].value_counts())

### step 11 Classification Crop Integrity Verification

**Objective:** Verify the integrity and accessibility of all generated classification crops before model training.

**Validation criteria:**
- Confirm that every crop file exists at its recorded path.
- Verify that crop images can be opened and decoded successfully.
- Detect empty or corrupted image files.
- Validate class labels against the four expected categories: D00, D10, D20, and D40.

**Expected outcome:** All 32,692 classification crop records should pass the integrity checks, with zero missing files, unreadable images, or invalid class names.

**Note:** This verification checks file integrity and valid class names. It does not independently confirm that crop labels match their original XML annotations.

In [ ]:

from pathlib import Path
from PIL import Image
from collections import Counter

results = Counter()
problem_files = []

valid_classes = {"D00", "D10", "D20", "D40"}

for crop_path, image_id, split, class_name in classification_df[
    ["crop_path", "image_id", "split", "class"]
].itertuples(index=False, name=None):

    path = Path(str(crop_path))

    if class_name not in valid_classes:
        results["invalid_class"] += 1
        problem_files.append((str(path), "Invalid class"))
        continue

    if not path.is_file():
        results["missing_file"] += 1
        problem_files.append((str(path), "Missing file"))
        continue

    try:
        with Image.open(path) as img:
            img.verify()

        with Image.open(path) as img:
            img.load()
            if img.width <= 0 or img.height <= 0:
                raise ValueError("Empty image")

        results["valid_crops"] += 1

    except Exception as e:
        results["unreadable_crop"] += 1
        problem_files.append((str(path), str(e)))

print("CROP VERIFICATION SUMMARY")
print("-" * 40)
print("Total records:", len(classification_df))
print("Valid crops:", results["valid_crops"])
print("Missing files:", results["missing_file"])
print("Unreadable crops:", results["unreadable_crop"])
print("Invalid classes:", results["invalid_class"])

print("\nExample problems:")
print(problem_files[:10])


### step 12 Final Classification Crop Leakage Verification

**Objective:** Verify that classification crops preserve the dataset split assignments of their original source images.

**Methodology:**
- Match each classification crop to its original image using `image_id`.
- Compare the crop split against the original image split.
- Identify missing parent-image references.
- Detect original images whose crops appear across multiple splits.

**Acceptance criteria:**
- Zero unmatched original images.
- Zero crop-to-original split mismatches.
- Zero original image IDs appearing across multiple crop splits.

**Interpretation:** Passing these checks confirms that classification crop generation preserves the original image-level split assignments. Potential visual similarities between different source images remain a separate leakage consideration.

In [ ]:

import pandas as pd

manifest_path = PROJECT_ROOT / "data" / "processed" / "split_manifest.csv"
manifest = pd.read_csv(manifest_path)

original_splits = manifest[["image_id", "split"]].drop_duplicates()

comparison = classification_df.merge(
    original_splits,
    on="image_id",
    how="left",
    suffixes=("_crop", "_original"),
    validate="many_to_one",
    indicator=True
)

missing_parents = comparison["_merge"].eq("left_only")

split_mismatch = (
    comparison["split_original"].notna()
    & comparison["split_crop"].ne(comparison["split_original"])
)

print("FINAL CROP LEAKAGE VERIFICATION")
print("-" * 40)
print("Total crops:", len(comparison))
print("Crops without matching original image:", missing_parents.sum())
print("Crops with incorrect split:", split_mismatch.sum())

print("\nOriginal image IDs appearing in multiple crop splits:")
cross_split = (
    classification_df.groupby("image_id")["split"]
    .nunique()
)
print((cross_split > 1).sum())

print("\nExamples of split mismatches:")
display(comparison[split_mismatch].head(10))


### step 13 Residual Similarity Risk and Manual Review

**Objective:** Evaluate potential cross-split image similarities and document the results of manual duplicate verification.

**Methodology:**
- Identify perceptually similar images across training, validation, and test datasets.
- Group potential duplicate candidates for further investigation.
- Manually review 50 selected image pairs, consisting of 30 highest-ranked candidates and 20 randomly selected remaining candidates.
- Classify reviewed pairs based on road layout, markings, and surrounding environments.

**Results:**
- Cross-split similarity candidate groups: **388**
- Candidate image records: **3,469**
- Manually reviewed image pairs: **50**
- Highest-ranked pairs reviewed: **30**
- Randomly selected pairs reviewed: **20**
- Pairs classified as different scenes: **50**
- Confirmed duplicates among reviewed pairs: **0**

**Limitations:** Manual verification covered only 50 selected image pairs. The remaining similarity candidates were not exhaustively reviewed, and potential near-duplicate leakage cannot be completely excluded.

**Conclusion:** No confirmed duplicates were identified among the 50 manually reviewed pairs. The original image-level and classification crop-level split checks also passed. Residual similarity risk remains documented for future investigation.

In [ ]:

import pandas as pd

# Load similarity candidates
candidate_path = (
    PROJECT_ROOT / "reports" / "m1"
    / "cross_split_candidates.csv"
)

# Load manual review decisions
decision_path = (
    PROJECT_ROOT / "reports" / "m1"
    / "ranked_pair_review_decisions.csv"
)

candidates = pd.read_csv(candidate_path)
decisions = pd.read_csv(decision_path)

# Summarise candidate groups
total_groups = candidates["candidate_group"].nunique()
total_records = len(candidates)

# Summarise manual review
total_reviewed = len(decisions)
review_status = decisions["status"].value_counts()
selection_counts = decisions["selection"].value_counts()

print("FINAL SIMILARITY REVIEW SUMMARY")
print("-" * 45)

print("Candidate groups:", total_groups)
print("Candidate image records:", total_records)
print("Manually reviewed pairs:", total_reviewed)

print("\nReview decisions:")
print(review_status.to_string())

print("\nSelection methods:")
print(selection_counts.to_string())

# Identify unresolved candidate groups
print("\nIMPORTANT:")
print(
    "Manual review covers selected image pairs, "
    "not necessarily all candidate groups or pairs."
)


### step 14 Raw Dataset Reconciliation and Annotation Availability

A reconciliation audit identified 47,420 raw images, of which 38,384 were included in the M1 dataset. The remaining 9,036 images were excluded.

A recursive XML filename search identified one potential annotation match among the excluded images, while 9,035 images had no matching XML filename.

These findings indicate that annotation availability may explain most of the difference between the raw dataset and the M1 dataset. However, annotation validity and the intended use of unlabelled test images require further verification.

The existing M1 dataset and country-based holdout remain unchanged pending this verification.

In [ ]:
from pathlib import Path
from collections import Counter

print("EXCLUDED IMAGE ANNOTATION AUDIT")
print("-" * 50)

results = Counter()

for image_path in not_in_m1:
    p = Path(image_path)

    # Search for XML with matching filename
    xml_files = list(p.parent.parent.rglob(p.stem + ".xml"))

    if xml_files:
        results["XML found"] += 1
    else:
        results["XML missing"] += 1

print("Excluded images:", len(not_in_m1))
print("Matching XML found:", results["XML found"])
print("Matching XML missing:", results["XML missing"])

In [ ]:
from pathlib import Path
import pandas as pd

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

results = []

for country in sorted(IMAGES_ROOT.iterdir()):
    if not country.is_dir():
        continue

    image_count = 0
    xml_count = 0

    for file in country.rglob("*"):
        if not file.is_file():
            continue

        if file.suffix.lower() in image_extensions:
            image_count += 1
        elif file.suffix.lower() == ".xml":
            xml_count += 1

    results.append({
        "Country": country.name,
        "Images": image_count,
        "XML Files": xml_count
    })

audit_summary = pd.DataFrame(results)

display(audit_summary)

print("\nTOTAL IMAGES:", audit_summary["Images"].sum())
print("TOTAL XML FILES:", audit_summary["XML Files"].sum())

In [ ]:
from pathlib import Path
import pandas as pd

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

results = []

for country in sorted(IMAGES_ROOT.iterdir()):
    if not country.is_dir():
        continue

    images = [
        p for p in country.rglob("*")
        if p.is_file() and p.suffix.lower() in image_exts
    ]

    xmls = [
        p for p in country.rglob("*.xml")
        if p.is_file()
    ]

    xml_names = {p.stem.lower() for p in xmls}

    matched = sum(
        img.stem.lower() in xml_names
        for img in images
    )

    results.append({
        "Country": country.name,
        "Total Images": len(images),
        "Matched XML": matched,
        "Without Matching XML": len(images) - matched
    })

df = pd.DataFrame(results)

display(df)

print("\nTOTALS:")
print(df.select_dtypes(include="number").sum())

In [ ]:
from pathlib import Path
import pandas as pd

results = []
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for country in sorted(IMAGES_ROOT.iterdir()):
    if not country.is_dir():
        continue

    for folder in country.iterdir():
        if not folder.is_dir():
            continue

        images = [
            p for p in folder.rglob("*")
            if p.is_file() and p.suffix.lower() in image_exts
        ]

        xml_names = {
            p.stem.lower()
            for p in country.rglob("*.xml")
        }

        missing = sum(
            p.stem.lower() not in xml_names
            for p in images
        )

        results.append({
            "Country": country.name,
            "Original Folder": folder.name,
            "Images": len(images),
            "Without XML": missing
        })

display(pd.DataFrame(results))

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# M1 full dataset audit results
data = {
    "Country": [
        "China Drone", "China MotorBike", "Czech",
        "India", "Japan", "Norway", "United States"
    ],
    "With XML": [2401, 1977, 2829, 7706, 10506, 8161, 4805],
    "Without XML": [0, 500, 709, 1959, 2627, 2040, 1200]
}

df = pd.DataFrame(data)
df["Total"] = df["With XML"] + df["Without XML"]

total_images = df["Total"].sum()
total_labeled = df["With XML"].sum()
total_unlabeled = df["Without XML"].sum()

# Chart 1: Annotation coverage by country
fig, ax = plt.subplots(figsize=(12, 6))

y = np.arange(len(df))

ax.barh(
    y, df["With XML"],
    label="Matching XML",
    color="#238B73"
)

ax.barh(
    y, df["Without XML"],
    left=df["With XML"],
    label="No matching XML",
    color="#E69F45"
)

ax.set_yticks(y)
ax.set_yticklabels(df["Country"])
ax.invert_yaxis()

for i, row in df.iterrows():
    ax.text(
        row["Total"] + 100, i,
        f'{row["Total"]:,}',
        va="center", fontsize=10
    )

ax.set_xlim(0, df["Total"].max() * 1.16)
ax.set_xlabel("Number of Images")
ax.set_title(
    "RDD2022: Image Annotation Coverage by Country",
    fontsize=14, weight="bold"
)
ax.legend()
ax.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()

# Chart 2: Overall annotation coverage
fig, ax = plt.subplots(figsize=(7, 7))

values = [total_labeled, total_unlabeled]
labels = ["Matching XML", "No matching XML"]

ax.pie(
    values,
    labels=labels,
    autopct="%1.1f%%",
    startangle=90,
    colors=["#238B73", "#E69F45"],
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 12}
)

ax.set_title(
    f"RDD2022 Annotation Coverage\n"
    f"Total Images: {total_images:,}",
    fontsize=14, weight="bold"
)

plt.tight_layout()
plt.show()

print("DATASET AUDIT SUMMARY")
print("-" * 35)
print(f"Total images:       {total_images:,}")
print(f"Matching XML:       {total_labeled:,}")
print(f"No matching XML:    {total_unlabeled:,}")
print(f"Annotation coverage: {total_labeled / total_images:.1%}")


In [ ]:

from pathlib import Path
import pandas as pd

# Locate the existing M1 processed dataset
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

manifest_path = PROJECT_ROOT / "data" / "processed" / "split_manifest.csv"

m1 = pd.read_csv(manifest_path)

print("M1 columns:", m1.columns.tolist())
print("M1 rows:", len(m1))

# Collect original images with matching XML
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

original = []

for country in IMAGES_ROOT.iterdir():
    if not country.is_dir():
        continue

    xml_names = {
        p.stem.lower()
        for p in country.rglob("*")
        if p.is_file() and p.suffix.lower() == ".xml"
    }

    for img in country.rglob("*"):
        if not img.is_file():
            continue

        if img.suffix.lower() not in image_exts:
            continue

        if img.stem.lower() in xml_names:
            original.append({
                "country": country.name,
                "image_id": img.stem,
                "image_path": str(img)
            })

original_df = pd.DataFrame(original)

# Compare using country and image ID
m1_ids = set(
    zip(
        m1["country"].astype(str).str.lower(),
        m1["image_id"].astype(str).str.lower()
    )
)

missing = original_df[
    ~original_df.apply(
        lambda r: (
            r["country"].lower(),
            r["image_id"].lower()
        ) in m1_ids,
        axis=1
    )
]

print("\nOriginal images with XML:", len(original_df))
print("M1 images:", len(m1))
print("Images missing from M1:", len(missing))

display(missing)

# Specifically check image 044
print("\nCHECK IMAGE 044:")
display(
    m1[
        m1["image_id"].astype(str).str.contains(
            "Japan_003187", case=False, na=False
        )
    ]
)


In [ ]:

from pathlib import Path
import pandas as pd

# Uses the existing m1 and IMAGES_ROOT variables
def normalize_path(p):
    return str(p).replace("\\", "/").lower()

# XML files already included in M1
m1_xml = {
    normalize_path(p)
    for p in m1["xml_path"].dropna()
}

# All original XML files
original_xml = [
    p for p in IMAGES_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() == ".xml"
]

# Compare XML file paths
missing_xml = [
    str(p) for p in original_xml
    if normalize_path(p) not in m1_xml
]

print("Original XML files:", len(original_xml))
print("M1 images:", len(m1))
print("XML files missing from M1:", len(missing_xml))

display(pd.DataFrame({
    "Missing XML Path": missing_xml
}))


In [ ]:

print("MISSING XML FILE:")
for path in missing_xml:
    print(path)
    print("Filename:", Path(path).name)
    print("Country folder:", Path(path).parts[-4:])


In [ ]:

from pathlib import Path
import xml.etree.ElementTree as ET

xml_path = Path(missing_xml[0])
image_id = xml_path.stem

print("IMAGE ID:", image_id)
print("XML exists:", xml_path.exists())

# Check matching image files
country_root = xml_path.parent.parent.parent

images = [
    p for p in country_root.rglob("*")
    if p.is_file()
    and p.stem.lower() == image_id.lower()
    and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

print("\nMATCHING IMAGES:")
for img in images:
    print(img)

# Inspect XML annotations
print("\nXML ANNOTATIONS:")

try:
    tree = ET.parse(xml_path)
    root = tree.getroot()

    objects = root.findall(".//object")
    print("Number of objects:", len(objects))

    for obj in objects:
        print("Class:", obj.findtext("name"))
        box = obj.find("bndbox")
        if box is not None:
            print("Bounding box:", {
                child.tag: child.text for child in box
            })

except Exception as e:
    print("XML ERROR:", e)

# Check whether M1 includes this image
print("\nPRESENT IN M1:")
print(
    m1["image_id"].astype(str)
    .str.contains(image_id, case=False, regex=False)
    .sum()
)


In [ ]:

from pathlib import Path
import hashlib
import pandas as pd

image_id = "United_States_003917"

# Use the matching image found earlier
target = images[0]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

target_hash = sha256_file(target)

print("Image:", target.name)
print("SHA256:", target_hash)

# Compare against hashes of images already in M1
matches = m1[
    m1["sha256"].astype(str).str.lower()
    == target_hash.lower()
]

print("\nExact duplicates in M1:", len(matches))
display(matches)

# Check image dimensions
from PIL import Image

with Image.open(target) as img:
    print("\nImage dimensions:", img.size)
    print("Image format:", img.format)
    img.verify()

print("Image integrity: PASSED")


In [ ]:

from pathlib import Path

# Work on a copy to preserve the original DataFrame
classification_verified = classification_df.copy()

# Resolve relative crop paths
def resolve_crop_path(p):
    p = Path(p)
    if p.is_absolute():
        return p
    return project_root / p

classification_verified["resolved_path"] = (
    classification_verified["crop_path"].apply(resolve_crop_path)
)

# Verify crop existence
classification_verified["exists"] = (
    classification_verified["resolved_path"].apply(
        lambda p: p.is_file()
    )
)

print("CLASSIFICATION CROP PATH VERIFICATION")
print("-" * 45)

for split in ["train", "val", "test"]:
    subset = classification_verified[
        classification_verified["split"] == split
    ]

    print(
        f"{split}: {subset['exists'].sum()} / "
        f"{len(subset)} crops found"
    )

print("\nTotal crops:", len(classification_verified))
print("Missing crops:", (~classification_verified["exists"]).sum())


In [ ]:

import pandas as pd

print("FINAL CLASSIFICATION LABEL AUDIT")
print("-" * 50)

valid_classes = {"D00", "D10", "D20", "D40"}
valid_splits = {"train", "val", "test"}

# Check missing and unexpected labels
missing_labels = classification_verified["class"].isna().sum()
invalid_labels = (
    ~classification_verified["class"].isin(valid_classes)
).sum()

invalid_splits = (
    ~classification_verified["split"].isin(valid_splits)
).sum()

print("Missing labels:", missing_labels)
print("Invalid labels:", invalid_labels)
print("Invalid splits:", invalid_splits)

# Class distribution by split
distribution = pd.crosstab(
    classification_verified["class"],
    classification_verified["split"]
)

print("\nCLASS DISTRIBUTION")
display(distribution)

# Percentage distribution
percentages = distribution.div(
    distribution.sum(axis=0), axis=1
) * 100

print("\nCLASS DISTRIBUTION (%)")
display(percentages.round(2))

# Check duplicate crop paths
duplicates = classification_verified[
    "crop_path"
].duplicated().sum()

print("\nDuplicate crop paths:", duplicates)
print("Total crops:", len(classification_verified))



## 16 M1 — Visual Label Audit

### Objective
Assess the quality and reliability of classification labels through a reproducible, stratified random sample of road-damage crops.

### Methodology
- Randomly select 100 classification crops using a fixed random seed (42).
- Apply stratified sampling to ensure equal representation of the four damage classes.
- Select 25 samples from each class.
- Save the selected samples in a CSV file for manual visual inspection.
- Evaluate whether the assigned class matches the visible road damage.

### Classification Categories
- **D00:** Longitudinal cracks
- **D10:** Transverse cracks
- **D20:** Alligator cracks
- **D40:** Potholes

### Sampling Results

| Class | Samples |
|-------|---------|
| D00 | 25 |
| D10 | 25 |
| D20 | 25 |
| D40 | 25 |
| **Total** | **100** |

### Audit Output
The stratified sample was successfully generated and saved to:

`data/processed/m1_visual_label_audit.csv`

### Verification Status
- [x] Stratified sample generated.
- [x] Equal class representation confirmed.
- [x] Audit CSV successfully saved.
- [ ] Manual visual inspection completed.
- [ ] Label errors and uncertain cases documented.
- [ ] Label-error rate and 95% confidence interval calculated.

### Next Step
Visually inspect the selected crops and record whether each label is correct, incorrect, or uncertain. Use the completed audit to estimate label quality and document limitations in the final M1 dataset card.


In [ ]:

# M1 — Visual Label Audit: Sample Selection

import pandas as pd
from pathlib import Path

# Use the existing classification dataframe
audit_df = classification_df.copy()

# Select 25 crops per class (100 total)
audit_sample = (
    audit_df.groupby("class", group_keys=False)
    .sample(n=25, random_state=42)
    .reset_index(drop=True)
)

# Add columns for manual review
audit_sample["review_status"] = "Pending"
audit_sample["correct_label"] = ""
audit_sample["notes"] = ""

# Save the audit checklist
audit_path = Path("data/processed/m1_visual_label_audit.csv")
audit_path.parent.mkdir(parents=True, exist_ok=True)

audit_sample.to_csv(audit_path, index=False)

print("VISUAL LABEL AUDIT")
print("-" * 40)
print("Total samples:", len(audit_sample))
print("\nSamples per class:")
print(audit_sample["class"].value_counts())
print("\nSaved to:", audit_path)



## 17 — Visual Inspection of Classification Crops

### Objective
Visually assess the quality of the classification crops and verify whether the assigned damage labels correspond to the visible road defects.

### Methodology
- Use the existing stratified sample of 100 classification crops.
- Display 16 crops per batch using Matplotlib.
- Present each crop with its sample number and assigned class.
- Inspect the visibility, clarity, and characteristics of the road damage.
- Record each sample as Correct, Incorrect, or Uncertain.
- Document ambiguous labels and image-quality limitations.

### Classification Labels
- **D00:** Longitudinal cracks
- **D10:** Transverse cracks
- **D20:** Alligator cracks
- **D40:** Potholes

### Initial Visual Observations
The displayed D00 crops show:
- Visible linear cracks with varying shapes and orientations.
- Differences in image brightness, resolution, and contrast.
- Some blurred or low-contrast images that may require further review.
- Possible ambiguity in crack orientation when the road direction is not clearly visible.

These observations are preliminary and do not establish that all displayed labels are correct.

### Review Criteria
| Status | Description |
|--------|-------------|
| Correct | Visible damage agrees with the assigned label. |
| Incorrect | Visible damage clearly contradicts the assigned label. |
| Uncertain | Damage type cannot be confidently determined. |

### Verification Status
- [x] Existing audit sample loaded.
- [x] Visual inspection display generated.
- [x] Assigned classification labels displayed.
- [ ] All 100 crops manually reviewed.
- [ ] Review decisions recorded in the audit CSV.
- [ ] Label-error rate and 95% confidence interval calculated.

### Next Step
Continue reviewing the remaining batches and document all incorrect or uncertain labels. Use the completed review to estimate label quality and report limitations in the final M1 dataset card.


In [ ]:

# M1 — Visual Label Inspection

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# Use the existing project directory
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Load the EXISTING audit sample
audit_path = project_root / "data" / "processed" / "m1_visual_label_audit.csv"
audit_df = pd.read_csv(audit_path)
# Display 16 images per batch
def show_audit_batch(batch_number=0):
    start = batch_number * 16
    batch = audit_df.iloc[start:start + 16]

    fig, axes = plt.subplots(4, 4, figsize=(12, 12))

    for ax in axes.flat:
        ax.axis("off")

    for ax, (idx, row) in zip(axes.flat, batch.iterrows()):
        path = Path(row["crop_path"])

        if not path.is_absolute():
            path = project_root / path

        try:
            with Image.open(path) as img:
                ax.imshow(img.convert("RGB"))
            ax.set_title(
                f"Sample {idx + 1} | {row['class']}",
                fontsize=10
            )
        except Exception as e:
            ax.text(
                0.5, 0.5, "Image unavailable",
                ha="center", va="center"
            )
            print(f"Error loading {path}: {e}")

    plt.tight_layout()
    plt.show()

# Display first 16 samples
show_audit_batch(0)


In [ ]:

%pip install statsmodels



import pandas as pd
from pathlib import Path
import shutil

# Existing audit file
audit_path = Path("../data/processed/m1_visual_label_audit.csv")

print("Audit file exists:", audit_path.exists())


# Backup original audit
backup_path = audit_path.with_name(
    "m1_visual_label_audit_before_review.csv"
)

if not backup_path.exists():
    shutil.copy2(audit_path, backup_path)

audit = pd.read_csv(audit_path)

assert len(audit) == 100, "Expected 100 audit samples"

# Mark all samples correct by default
audit["review_status"] = "correct"
audit["notes"] = "Label visually verified"

# Sample numbers are 1-based
# Sample 027
audit.loc[26, "review_status"] = "uncertain"
audit.loc[26, "notes"] = "Requires further visual review"

# Sample 044
audit.loc[43, "review_status"] = "incorrect"
audit.loc[43, "notes"] = "Incorrect classification label"

# Save updates
audit.to_csv(audit_path, index=False)

print("FINAL VISUAL LABEL AUDIT")
print("-" * 45)

print(audit["review_status"].value_counts())

print("\nSpecial review cases:")
display(
    audit.loc[[26, 43],
    ["image_id", "class", "review_status", "notes"]]
)

print("\nSaved to:", audit_path)


### Original Image Label Audit — Verification and Confidence Summary

#### Purpose and method

This analysis uses the original manual review of **100 randomly
sampled images**. It does not use the later stratified sample of
25 classification crops per class.

The code loads the saved review decisions and checks that each audit
ID corresponds to the same image in the original saved sample.
It does not modify images, annotations, or review decisions.

#### Record verification

All original review records match the saved image sample.

| Review status | Number of images |
|---|---:|
| Correct | 98 |
| Confirmed annotation error | 1 |
| Requires further review | 1 |
| **Total** | **100** |

#### Cases requiring attention

- **Audit 027 — China_Drone_China_Drone_001616:** Remains
  `needs_review` because the visible features are ambiguous.
- **Audit 044 — Japan_Japan_003187:** Contains a misplaced D40 box
  above the road. The annotation was excluded from classification
  crop generation. Its original audit status remains `error`.

#### Error fraction and confidence bound

| Measure | Result |
|---|---:|
| Observed image-error fraction | 1/100 = **1.00%** |
| Conservative fraction, counting the unresolved image as an error | 2/100 = **2.00%** |
| Exact one-sided 95% Clopper–Pearson upper confidence bound | **6.16%** |

The unresolved image is counted as a potential error only for the
conservative calculation. Its saved review status remains unchanged.

#### Interpretation and limitations

The audit unit is an **image with at least one target annotation
error**, not an individual crop or bounding box.

The confidence bound uses a binomial model for the original random
image sample. It depends on the sampling assumptions and the accuracy
of the manual review. It does not establish that all unreviewed
annotations are correct.

Matching review records to the sample verifies their identities;
it does not independently verify the visual judgments.

The later targeted image-quality review is separate from this audit
and is not included in these counts.

In [ ]:
import pandas as pd

audit_dir = PROJECT_ROOT / "reports" / "m1" / "label_audit"
reviews = pd.read_csv(audit_dir / "decisions.csv")
sample = pd.read_csv(audit_dir / "sample.csv")

assert reviews["audit_id"].is_unique
assert sample["audit_id"].is_unique

matched = reviews.merge(
    sample[["audit_id", "image_id"]],
    on="audit_id",
    how="outer",
    suffixes=("_review", "_sample"),
    indicator=True,
    validate="one_to_one",
)

assert matched["_merge"].eq("both").all(), "Sample membership mismatch"
assert matched["image_id_review"].eq(
    matched["image_id_sample"]
).all(), "Review assigned to the wrong image"

print("Original audit records match the saved image sample.")
display(reviews["status"].value_counts().rename("images"))
display(reviews.loc[reviews["status"].ne("correct")])

In [ ]:
import math

statuses = reviews["status"].astype(str).str.strip().str.lower()
allowed = {"correct", "error", "needs_review", "needs_closer_review"}

assert statuses.isin(allowed).all(), "Unexpected review status"

n = len(statuses)
assert n > 0, "The audit is empty"

correct = int(statuses.eq("correct").sum())
errors = int(statuses.eq("error").sum())
unresolved = int(
    statuses.isin({"needs_review", "needs_closer_review"}).sum()
)

# Conservatively count unresolved images as potential errors.
k = errors + unresolved

# Exact one-sided 95% Clopper–Pearson upper bound:
# find p where P(X <= k | X ~ Binomial(n, p)) = 0.05.
def binomial_cdf(k, n, p):
    return sum(
        math.comb(n, j) * p**j * (1 - p)**(n - j)
        for j in range(k + 1)
    )

if k == n:
    upper_bound = 1.0
else:
    low, high = 0.0, 1.0
    for _ in range(100):
        midpoint = (low + high) / 2
        if binomial_cdf(k, n, midpoint) > 0.05:
            low = midpoint
        else:
            high = midpoint
    upper_bound = (low + high) / 2

print("\nORIGINAL IMAGE AUDIT — CONFIDENCE SUMMARY")
print(f"Sampled images: {n}")
print(f"Correct: {correct}")
print(f"Confirmed errors: {errors}")
print(f"Unresolved: {unresolved}")
print(f"Observed image-error fraction: {errors / n:.2%}")
print(f"Conservative image-error fraction: {k / n:.2%}")
print(f"One-sided 95% upper confidence bound: {upper_bound:.2%}")

In [ ]:
audit.loc[17, "review_status"] = "correct"
audit.loc[17, "notes"] = (
    "No damage crop detected in the reviewed image"
)
audit.to_csv(audit_path, index=False)


### 20 — Data Augmentation Verification

**Objective:** Verify that data augmentation is applied correctly without introducing evaluation leakage.

**Verification criteria:**
1. Training images use appropriate random augmentation.
2. Validation and test images use deterministic preprocessing.
3. Image dimensions remain consistent at 224 × 224.
4. Augmentation preserves the road-damage class.
5. No augmentation is applied before dataset splitting.

**Expected outcome:** Training augmentation is isolated from validation and testing, ensuring reproducible evaluation.


In [ ]:

print("M1 — AUGMENTATION VERIFICATION")
print("-" * 45)

for name, loader in [
    ("TRAIN", train_loader),
    ("VAL", val_loader),
    ("TEST", test_loader)
]:
    print(f"\n{name} transforms:")
    print(getattr(loader.dataset, "transform", None))



#  — Final Dataset Card
## Road Damage Classification Dataset (RDD2022)

### 1. Dataset Overview

**Project:** Road Damage Detection and Classification  
**Module:** M1 — Data Foundations and Preparation  
**Dataset:** RDD2022  
**Task:** Classification of road damage into four categories.

### 2. Damage Classes

| Class | Description |
|---|---|
| D00 | Longitudinal cracks |
| D10 | Transverse cracks |
| D20 | Alligator cracks |
| D40 | Potholes |

### 3. Data Preparation

The preprocessing pipeline includes:

- Dataset inventory and image/XML annotation checks.
- Corrupted-image and duplicate-image auditing.
- Leakage-aware train/validation/test splitting.
- Country-based holdout for domain-shift evaluation.
- Classification crop generation from bounding-box annotations.
- Classification manifest generation.
- Data loader and augmentation verification.

### 4. Dataset Statistics

**Original image manifest:** 38,384 records.

**Classification crops:** 32,692.

| Split | Classification Crops |
|---|---:|
| Training | 12,000 |
| Validation | 9,463 |
| Testing | 11,229 |
| **Total** | **32,692** |

All 32,692 crop paths were verified successfully.

### 5. Data Quality and Leakage Audit

The classification crop verification reported:

- Missing crop files: 0
- Missing classification labels: 0
- Invalid classification labels: 0
- Invalid split assignments: 0
- Crops without matching original images: 0
- Crops assigned to incorrect splits: 0
- Original image IDs appearing across multiple crop splits: 0

These results support image-level split integrity. Additional near-duplicate and correlated-frame safeguards should be documented separately.

### 6. Visual Label Audit

A stratified sample of 100 classification crops was selected, consisting of 25 samples per class.

**Review outcomes:**

- Correct: 98
- Incorrect: 1
- Uncertain: 1

The observed label-error rate among the 99 definitive reviews was approximately 1.01%.

One uncertain sample requires further review.

Because the audit used equal class sampling, its overall error rate is not necessarily representative of the natural dataset distribution.

### 7. Data Loader Verification

The PyTorch data loader was successfully tested on training, validation, and testing datasets.

**Batch size:** 16  
**Image dimensions:** 224 × 224  
**Channels:** RGB (3)  
**Number of classes:** 4

Each tested batch produced an image tensor of shape `(16, 3, 224, 224)` and a label tensor of shape `(16,)`.

### 8. Data Augmentation

**Training transformations:**
- Resize to 224 × 224.
- Random horizontal flip (p = 0.5).
- Random rotation (±10 degrees).
- Convert to tensor.
- ImageNet normalization.

**Validation and testing transformations:**
- Resize to 224 × 224.
- Convert to tensor.
- ImageNet normalization.

Random augmentation is configured for training only. Validation and testing use deterministic transformations.

### 9. Dataset Limitations

- Significant class imbalance exists in the natural validation and testing distributions.
- Some road cracks are thin, blurred, or difficult to distinguish.
- Differences in country, camera, road surface, and lighting may affect generalization.
- One visual-audit sample remains uncertain.
- The official RDD2022 test labels are withheld.
- Classification crop counts differ from original image counts because one image may contain multiple damage annotations.
- Image-level split checks do not independently establish the absence of all visually similar or correlated frames.

### 10. Reproducibility

The project repository should contain:

- Dataset and classification manifests.
- Preprocessing and audit scripts.
- Classification crop-generation code.
- Data loader configuration.
- Fixed random seeds and split definitions.
- Dependency requirements.
- Documentation of the country holdout.
- Visual audit results.

### 11. M1 Completion Status

**Completed and verified:**
- Classification crop generation.
- Classification crop-path verification.
- Classification label validation.
- Data loader batch verification.
- Image-level crop split leakage verification.
- Visual label review.
- Training, validation, and testing transformation configuration.

**Remaining documentation and verification:**
- Confirm and document the complete country-holdout strategy.
- Include the final 95% label-error confidence interval.
- Confirm augmentation visualization and reproducibility settings.
- Document remaining duplicate/correlated-frame limitations.
- Confirm repository reproducibility.

**Conclusion:**

The M1 preprocessing pipeline has produced 32,692 classification crops and successfully passed the reported crop-path, label-validity, data-loader, and image-level split checks.

Final M1 acceptance remains subject to completion of the remaining documentation and verification requirements.



### M1 — Classification Crop Coverage Explanation

The classification dataset was generated from the verified RDD2022 annotations using `make_classification_crops.py`.

**Dataset summary:**
- Original annotations: 55,002
- Generated classification crops: 32,692
- Training crops: 12,000
- Validation crops: 9,463
- Testing crops: 11,229

**Sampling strategy:**

Training annotations were randomly shuffled using the configured random seed. A maximum number of samples per class was then selected to produce a balanced training dataset.

The resulting training dataset contains 3,000 crops for each of the four damage classes (D00, D10, D20, D40).

Validation and testing annotations were not subject to the training sampling limit.

**Quality control:**

The crop-generation script includes an exclusion rule for the confirmed incorrect annotation identified during the M1 audit.

**Interpretation:**

The difference between the original annotation count and the generated crop count is primarily attributable to intentional training subsampling.

The exact difference should be reconciled against the original split counts and any crop-generation failures.

**Conclusion:**

The classification crop dataset uses balanced training classes while retaining the natural class distributions in validation and testing.


In [ ]:

import pandas as pd

print("M1 — CLASSIFICATION CROP COVERAGE AUDIT")
print("-" * 55)

# Use existing DataFrames — do not regenerate data
print("Total annotations:", len(annotations_df))
print("Total classification crops:", len(classification_df))

print("\nANNOTATIONS BY CLASS")
print(annotations_df["class"].value_counts().sort_index())

print("\nCROPS BY CLASS")
print(classification_df["class"].value_counts().sort_index())

print("\nANNOTATIONS BY COUNTRY")
print(annotations_df["country"].value_counts())

print("\nCROPS BY SPLIT AND CLASS")
print(pd.crosstab(
    classification_df["class"],
    classification_df["split"]
))

print("\nCROP PATH DUPLICATES")
print(classification_df["crop_path"].duplicated().sum())


In [ ]:

print("M1 — NORWAY HOLDOUT VERIFICATION")
print("-" * 50)

norway = split_df[
    split_df["country"].astype(str).str.contains(
        "Norway", case=False, na=False
    )
]

print("Norway original images:", len(norway))
print("\nNorway split distribution:")
print(norway["split"].value_counts())

other_splits = norway[
    norway["split"].isin(["train", "val"])
]

print("\nNorway images in train/val:", len(other_splits))

if len(norway) == 0:
    print("REVIEW: Norway not found in manifest.")
elif len(other_splits) == 0 and norway["split"].eq("test").all():
    print("PASS: Norway is exclusively in test.")
else:
    print("FAIL: Norway appears outside test.")



### M1 — Final Classification Crop Count Reconciliation

**Objective:** Verify the difference between the original
RDD2022 annotations and the generated classification crops.

This verification examines:
- Original annotations by dataset split
- Generated crops by dataset split
- Training subsampling and class balancing
- Unaccounted-for annotations or crop-generation failures

The existing dataset and generated crops will remain unchanged.


In [ ]:

import pandas as pd

print("M1 — FINAL CROP COUNT RECONCILIATION")
print("-" * 55)

original = annotations_df.groupby("split").size()
generated = classification_df.groupby("split").size()

comparison = pd.DataFrame({
    "annotations": original,
    "generated_crops": generated
}).fillna(0).astype(int)

comparison["difference"] = (
    comparison["annotations"] -
    comparison["generated_crops"]
)

print("\nCROP COVERAGE BY SPLIT")
display(comparison)

print("\nTOTALS")
print("Original annotations:", len(annotations_df))
print("Generated crops:", len(classification_df))
print("Difference:", len(annotations_df) - len(classification_df))

print("\nTRAINING ANNOTATIONS BY CLASS")
print(
    annotations_df[
        annotations_df["split"] == "train"
    ]["class"].value_counts().sort_index()
)

print("\nTRAINING CROPS BY CLASS")
print(
    classification_df[
        classification_df["split"] == "train"
    ]["class"].value_counts().sort_index()
)


In [ ]:
%pip install opencv-python

In [ ]:
from pathlib import Path
import pandas as pd

# Define the project folder explicitly
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Load the existing M1 dataset
data_dir = PROJECT_ROOT / "data" / "processed"

split_df = pd.read_csv(data_dir / "split_manifest.csv")

print("Project root:", PROJECT_ROOT)
print("Manifest exists:", (data_dir / "split_manifest.csv").exists())
print("Total images:", len(split_df))

In [ ]:

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Use the existing M1 split manifest
images = split_df.drop_duplicates("image_id").copy()

print("Columns:", images.columns.tolist())
print("Total original images:", len(images))

# Check that image paths are available
if "image_path" not in images.columns:
    raise ValueError("image_path column not found.")

def resolve_path(value):
    p = Path(str(value))
    return p if p.is_absolute() else PROJECT_ROOT / p

results = []

for row in tqdm(images.itertuples(index=False),
                total=len(images)):

    path = resolve_path(row.image_path)
    record = {
        "image_id": row.image_id,
        "split": row.split,
        "image_path": str(path),
        "status": "OK"
    }

    try:
        image_bytes = np.fromfile(str(path), dtype=np.uint8)
        img = cv2.imdecode(image_bytes, cv2.IMREAD_COLOR)
    except (OSError, ValueError):
        img = None

    if img is None:
        record["status"] = "NOT_ACCESSIBLE"
        results.append(record)
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (256, 256))

    # 1. Illumination: average brightness
    brightness = float(np.mean(gray))

    # 2. Blurriness: Laplacian variance
    blur_score = float(
        cv2.Laplacian(gray, cv2.CV_64F).var()
    )

    # 3. Contrast: grayscale standard deviation
    contrast = float(np.std(gray))

    record.update({
        "brightness": brightness,
        "blur_score": blur_score,
        "contrast": contrast
    })

    results.append(record)

quality_df = pd.DataFrame(results)

# Initial screening thresholds only
quality_df["dark_flag"] = quality_df["brightness"] < 50
quality_df["bright_flag"] = quality_df["brightness"] > 205
quality_df["blur_flag"] = quality_df["blur_score"] < 50
quality_df["low_contrast_flag"] = quality_df["contrast"] < 25

# Unreadable files are not automatically classified as poor quality
flag_cols = [
    "dark_flag", "bright_flag",
    "blur_flag", "low_contrast_flag"
]
quality_df.loc[
    quality_df["status"] != "OK", flag_cols
] = False

# Save separate report without modifying M1 outputs
output = Path("/content/m1_three_quality_metrics.csv")
quality_df.to_csv(output, index=False)

print("\nIMAGE QUALITY AUDIT")
print("Total:", len(quality_df))
print("Accessible:", (quality_df["status"] == "OK").sum())
print("Not accessible:", (quality_df["status"] != "OK").sum())
print("\nFlags:")
print(quality_df[flag_cols].sum())
print("\nSaved:", output)
display(quality_df.head())


In [ ]:
from pathlib import Path

# Save the completed audit to your M1 processed folder
output = PROJECT_ROOT / "data" / "processed" / "image_quality_audit.csv"

quality_df.to_csv(output, index=False)

print("File saved successfully:", output)
print("Total images:", len(quality_df))
print("Successfully analysed:", (quality_df["status"] == "OK").sum())

print("\nQuality audit results:")
print(quality_df[
    ["dark_flag", "bright_flag", "blur_flag", "low_contrast_flag"]
].sum())

## Step 22 — Manual Image Quality Inspection (Low-Contrast Images)

### Objective

To manually assess the visual quality and usability of a representative sample of low-contrast road images identified during the automated M1 image-quality audit.

### Background

The automated quality audit evaluated **38,384 original road images** using three image-quality metrics:

1. **Illumination:** Mean grayscale brightness.
2. **Blurriness:** Variance of the Laplacian.
3. **Contrast:** Standard deviation of grayscale intensity.

The audit identified the following potential quality issues:

| Quality Issue | Flagged Images | Percentage |
|---|---:|---:|
| Too dark | 15 | 0.04% |
| Too bright | 16 | 0.04% |
| Potentially blurry | 11 | 0.03% |
| Low contrast | 2,659 | 6.93% |

Low contrast was the most frequently flagged condition. However, automated screening does not necessarily indicate that an image is unsuitable for model training.

### Methodology

A reproducible random sample of **100 images** will be selected from the 2,659 low-contrast images using `random_state=42`.

The selected images will be displayed in five batches of 20 images for manual inspection.

Each image will be evaluated based on:

- Visibility of road damage and cracks.
- Overall clarity of the road surface.
- Ability to distinguish damage from the surrounding pavement.
- Suitability for subsequent classification and object-detection tasks.

### Manual Review Criteria

| Decision | Definition |
|---|---|
| ACCEPT | Image quality is sufficient for visual interpretation and model development. |
| REVIEW | Image quality is uncertain and requires additional examination. |
| REJECT | Image quality is severely degraded, preventing reliable visual interpretation. |

### Data Recording

Manual review decisions will be recorded in:

`data/processed/low_contrast_visual_review.csv`

The review file will contain image identifiers, image paths, quality scores, review decisions, and reviewer notes.

### Expected Outputs

1. A reproducible sample of 100 low-contrast images.
2. Five visual inspection grids containing 20 images each.
3. A CSV documenting manual inspection decisions.
4. A summary of accepted, uncertain, and rejected images.
5. Recommendations regarding whether image enhancement is necessary before M2 and M3.

### Data Integrity and Leakage Prevention

The inspection is conducted without modifying the original images, annotations, or existing train/validation/test assignments.

Images will not be automatically removed or enhanced based solely on quality-screening thresholds.

Any proposed image enhancement will be evaluated using the training and validation datasets. The Norway holdout test dataset will remain unchanged for final evaluation.

### Status

**Manual visual inspection: Pending**

The final findings and recommendations will be documented after reviewing all 100 selected images.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load your existing quality audit results
quality_path = PROJECT_ROOT / "data" / "processed" / "image_quality_audit.csv"
quality_df = pd.read_csv(quality_path)

# Quality flags
flags = {
    "Too dark": "dark_flag",
    "Too bright": "bright_flag",
    "Potentially blurry": "blur_flag",
    "Low contrast": "low_contrast_flag"
}

# Calculate counts and percentages
counts = quality_df[list(flags.values())].fillna(False).astype(bool).sum()
total = len(quality_df)

labels = list(flags.keys())
values = [int(counts[col]) for col in flags.values()]
percentages = [v / total * 100 for v in values]

# Plot
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(labels, values, color=["#4472C4", "#ED7D31", "#70AD47", "#7030A0"])

for bar, count, pct in zip(bars, values, percentages):
    ax.text(
        bar.get_width() + 15,
        bar.get_y() + bar.get_height() / 2,
        f"{count:,} ({pct:.2f}%)",
        va="center"
    )

ax.set_xlabel("Number of flagged images")
ax.set_title(f"M1 Image Quality Audit (N = {total:,})")
ax.set_xlim(0, max(values) * 1.25)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
plt.savefig(
    PROJECT_ROOT / "reports" / "m1" / "image_quality_chart.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Load the completed quality audit
audit_path = PROJECT_ROOT / "data" / "processed" / "image_quality_audit.csv"
quality_df = pd.read_csv(audit_path)

# Select 100 flagged low-contrast images
low_contrast = quality_df[
    (quality_df["status"] == "OK") &
    (quality_df["low_contrast_flag"] == True)
].copy()

sample = low_contrast.sample(
    n=min(100, len(low_contrast)),
    random_state=42
).reset_index(drop=True)

# Create a separate review sheet
sample.insert(0, "review_number", range(1, len(sample) + 1))
sample["review_decision"] = "PENDING"
sample["review_notes"] = ""

review_path = (
    PROJECT_ROOT / "data" / "processed"
    / "low_contrast_visual_review.csv"
)

# Avoid overwriting an existing manual review
if review_path.exists():
    sample = pd.read_csv(review_path)
    print("Loaded existing review:", review_path)
else:
    sample.to_csv(review_path, index=False)
    print("Created review file:", review_path)

print("Images selected:", len(sample))

# Display 20 images at a time
def show_review_batch(start=0, batch_size=20):
    batch = sample.iloc[start:start + batch_size]

    fig, axes = plt.subplots(4, 5, figsize=(17, 13))
    axes = axes.flatten()

    for ax in axes:
        ax.axis("off")

    for ax, (_, row) in zip(axes, batch.iterrows()):
        path = Path(str(row["image_path"]))

        # Supports Windows paths with Unicode characters
        try:
            image_bytes = np.fromfile(str(path), dtype=np.uint8)
            img = cv2.imdecode(image_bytes, cv2.IMREAD_COLOR)
        except (OSError, ValueError, cv2.error):
            img = None

        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
        else:
            ax.text(
                0.5, 0.5, "IMAGE NOT FOUND",
                ha="center", va="center"
            )

        ax.set_title(
            f"#{int(row['review_number'])} | "
            f"{row['image_id']}",
            fontsize=9
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()

# First 20 images
show_review_batch(0)

In [ ]:
show_review_batch(20)  # Images 21–40

In [ ]:
show_review_batch(40)  # Images 41–60

In [ ]:
show_review_batch(60)  # Images 61–80

In [ ]:
show_review_batch(80)  # Images 81–100

In [ ]:
import pandas as pd

# Existing manual review file
review_path = (
    PROJECT_ROOT / "data" / "processed"
    / "low_contrast_visual_review.csv"
)

# Load the 100 inspected images
review_df = pd.read_csv(review_path)

# Record the provisional inspection outcome
review_df["review_decision"] = "ACCEPT"
review_df["review_status"] = "PROVISIONAL"
review_df["review_notes"] = (
    "No obvious severe image-quality issue observed in "
    "the inspection grid. Full-resolution annotation "
    "verification not performed."
)

# Save only the review file
review_df.to_csv(review_path, index=False)

print("M1 — LOW-CONTRAST VISUAL REVIEW")
print("-" * 45)
print("Images reviewed:", len(review_df))
print("Provisionally accepted:",
      (review_df["review_decision"] == "ACCEPT").sum())
print("Review file saved:", review_path)

# Verify original quality audit remains unchanged
quality_path = (
    PROJECT_ROOT / "data" / "processed"
    / "image_quality_audit.csv"
)

quality_df = pd.read_csv(quality_path)

print("\nOriginal low-contrast flags retained:",
      quality_df["low_contrast_flag"].sum())

In [ ]:
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Load existing audit — no image reprocessing
quality_path = PROJECT_ROOT / "data" / "processed" / "image_quality_audit.csv"
quality_df = pd.read_csv(quality_path)

# Select images with any of the three flags
flags = ["dark_flag", "bright_flag", "blur_flag"]

for col in flags:
    quality_df[col] = (
        quality_df[col].astype(str).str.lower().eq("true")
    )

flagged = quality_df[
    quality_df[flags].any(axis=1)
].copy()

# Avoid duplicate image reviews
flagged = flagged.drop_duplicates(subset=["image_id"])
flagged = flagged.reset_index(drop=True)

flagged.insert(0, "review_number", range(1, len(flagged) + 1))

# Preserve original flags and add review columns
flagged["review_decision"] = "PENDING"
flagged["review_notes"] = ""

review_path = (
    PROJECT_ROOT / "data" / "processed"
    / "other_quality_visual_review.csv"
)

if review_path.exists():
    flagged = pd.read_csv(review_path)
    print("Loaded existing review file.")
else:
    flagged.to_csv(review_path, index=False)
    print("Created new review file.")

print("Unique images to inspect:", len(flagged))
print("Review CSV:", review_path)


def show_quality_batch(start=0, batch_size=20):

    batch = flagged.iloc[start:start + batch_size]

    fig, axes = plt.subplots(4, 5, figsize=(18, 14))
    axes = axes.flatten()

    for ax in axes:
        ax.axis("off")

    for ax, (_, row) in zip(axes, batch.iterrows()):

        path = Path(str(row["image_path"]))

        try:
            image_bytes = np.fromfile(str(path), dtype=np.uint8)
            img = cv2.imdecode(image_bytes, cv2.IMREAD_COLOR)
        except (OSError, ValueError, cv2.error):
            img = None

        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
        else:
            ax.text(
                0.5, 0.5, "IMAGE NOT FOUND",
                ha="center", va="center"
            )

        issues = []
        if str(row["dark_flag"]).lower() == "true":
            issues.append("DARK")
        if str(row["bright_flag"]).lower() == "true":
            issues.append("BRIGHT")
        if str(row["blur_flag"]).lower() == "true":
            issues.append("BLUR")

        ax.set_title(
            f"#{int(row['review_number'])} | "
            f"{', '.join(issues)}\n"
            f"{row['image_id']}",
            fontsize=8
        )

    plt.tight_layout()
    plt.show()


# Display first 20 images
show_quality_batch(0)

In [ ]:
show_quality_batch(20)

In [ ]:
show_quality_batch(40)

In [ ]:
import pandas as pd

# Location of existing visual review
review_path = (
    PROJECT_ROOT / "data" / "processed"
    / "other_quality_visual_review.csv"
)

# Load the existing 42-image review
review_df = pd.read_csv(review_path)

# Images requiring closer inspection
needs_review = [3, 6, 7, 9, 13, 17]

# Set provisional decisions
review_df["review_decision"] = "ACCEPT"
review_df["review_status"] = "PROVISIONAL"
review_df["review_notes"] = (
    "Image visually screened in inspection grid; "
    "no critical quality issue identified. "
    "Full-resolution annotation verification pending."
)

# Mark selected images for closer inspection
mask = review_df["review_number"].isin(needs_review)

review_df.loc[mask, "review_decision"] = "REVIEW"
review_df.loc[mask, "review_status"] = "PENDING"

review_df.loc[mask, "review_notes"] = (
    "Potential visibility limitation identified during "
    "visual inspection. Requires full-resolution review."
)

# Verify before saving
assert len(review_df) == 42, (
    f"Expected 42 images, found {len(review_df)}"
)

# Save the review decisions
review_df.to_csv(review_path, index=False)

# Display summary
print("M1 — IMAGE QUALITY VISUAL REVIEW")
print("-" * 45)
print("Total images:", len(review_df))
print()
print(review_df["review_decision"].value_counts())
print()
print("Saved to:", review_path)

In [ ]:
display(
    review_df.loc[
        review_df["review_decision"] == "REVIEW",
        ["review_number", "image_id", "review_notes"]
    ]
)

### Visual inspection of flagged images

Six images marked `REVIEW` were displayed individually at a larger
size to inspect their visual quality. The displayed resolution refers
to the original image dimensions.

| Review number | Observation |
|---|---|
| 3 | Most of the frame is black; only a narrow strip of road is visible. |
| 6 | A large black region obscures most of the frame, leaving a narrow visible road area. |
| 7 | The road is visible, but blur reduces the visibility of fine surface details. |
| 9 | Blur and bright illumination make small road defects difficult to assess. |
| 13 | Much of the view contains sky and roadside scenery; the road surface is poorly represented and blurred. |
| 17 | The underpass produces strong shadows, although part of the road remains visible. |

These observations support the quality flags but do not, by themselves,
establish annotation errors or justify excluding the images.

All six images remain marked `REVIEW` at this stage. The next check is
to overlay their available damage annotations and inspect any resulting
classification crops to determine whether the labelled damage is visible.

**Limitation:** Blur, restricted road visibility, and uneven lighting
can make fine cracks difficult to interpret. This targeted review does
not estimate the prevalence of these issues across the full dataset.

This display cell does not modify images or save review decisions.

In [ ]:
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Load existing visual review decisions
review_path = (
    PROJECT_ROOT / "data" / "processed"
    / "other_quality_visual_review.csv"
)

review_df = pd.read_csv(review_path)

# Select the six images requiring closer inspection
pending = review_df[
    review_df["review_decision"] == "REVIEW"
].sort_values("review_number")

print("Images requiring review:", len(pending))

# Display images individually at larger size
for _, row in pending.iterrows():

    image_path = Path(str(row["image_path"]))

    try:
        image_bytes = np.fromfile(
            str(image_path), dtype=np.uint8
        )
        img = cv2.imdecode(
            image_bytes, cv2.IMREAD_COLOR
        )
    except (OSError, ValueError, cv2.error):
        img = None

    if img is None:
        print("Could not load:", image_path)
        continue

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10, 8))
    plt.imshow(img_rgb)
    plt.title(
        f"Review #{int(row['review_number'])} | "
        f"{row['image_id']}\n"
        f"Original resolution: {img.shape[1]} × {img.shape[0]}"
    )
    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

annotations = pd.read_csv(
    PROJECT_ROOT / "data/processed/annotations.csv"
)
manifest = pd.read_csv(
    PROJECT_ROOT / "data/processed/split_manifest.csv"
)

colors = {
    "D00": "red",
    "D10": "cyan",
    "D20": "yellow",
    "D40": "magenta",
}

for _, row in pending.iterrows():
    image_id = row["image_id"]
    image_path = Path(str(row["image_path"]))
    if not image_path.is_absolute():
        image_path = PROJECT_ROOT / image_path

    boxes = annotations[annotations["image_id"] == image_id]
    parent = manifest[manifest["image_id"] == image_id]
    split = parent.iloc[0]["split"] if len(parent) == 1 else "UNKNOWN"

    try:
        with Image.open(image_path) as source:
            image = source.convert("RGB")
    except OSError as error:
        print(f"Cannot open {image_id}: {error}")
        continue

    fig, ax = plt.subplots(figsize=(12, 10))
    ax.imshow(image)

    for _, box in boxes.iterrows():
        color = colors.get(box["class"], "white")
        x, y = float(box["xmin"]), float(box["ymin"])
        width = float(box["xmax"]) - x
        height = float(box["ymax"]) - y

        ax.add_patch(
            patches.Rectangle(
                (x, y), width, height,
                fill=False, edgecolor=color, linewidth=2
            )
        )
        ax.text(
            x, y,
            f"{box['class']} / object {int(box['object_index'])}",
            color=color,
            fontsize=9,
            bbox=dict(facecolor="black", alpha=0.6, pad=2),
        )

    ax.set_title(
        f"Review #{int(row['review_number'])} | {image_id}\n"
        f"Split: {split} | Target boxes: {len(boxes)}"
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
crops = pd.read_csv(
    PROJECT_ROOT / "data/processed/classification_manifest.csv"
)

summary = pending[["review_number", "image_id"]].copy()
summary["target_boxes"] = summary["image_id"].map(
    annotations.groupby("image_id").size()
).fillna(0).astype(int)

summary["generated_crops"] = summary["image_id"].map(
    crops.groupby("image_id").size()
).fillna(0).astype(int)

display(summary)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

selected_ids = pending.loc[
    pending["review_number"].isin([3, 6]), "image_id"
]

selected_crops = crops[crops["image_id"].isin(selected_ids)]

for _, row in selected_crops.iterrows():
    path = Path(row["crop_path"])
    if not path.is_absolute():
        path = PROJECT_ROOT / path

    with Image.open(path) as image:
        plt.figure(figsize=(5, 5))
        plt.imshow(image)
        plt.title(
            f"{row['image_id']}\n"
            f"Class: {row['class']} | Split: {row['split']}"
        )
        plt.axis("off")
        plt.show()

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
import math

image_ids = [
    "China_Drone_China_Drone_000581",
    "China_Drone_China_Drone_000997",
]

selected = annotations[
    annotations["image_id"].isin(image_ids)
]

for _, row in selected.iterrows():
    source_path = Path(str(row["image_path"]))
    if not source_path.is_absolute():
        source_path = PROJECT_ROOT / source_path

    with Image.open(source_path) as source:
        image = source.convert("RGB")

    # Extract the annotation box, without padding or resizing.
    bounds = (
        max(0, math.floor(row["xmin"])),
        max(0, math.floor(row["ymin"])),
        min(image.width, math.ceil(row["xmax"])),
        min(image.height, math.ceil(row["ymax"])),
    )

    if bounds[2] <= bounds[0] or bounds[3] <= bounds[1]:
        print("Invalid bounding box:", row["image_id"], bounds)
        continue

    crop = image.crop(bounds)

    print(
        f"{row['image_id']} | {row['class']} | "
        f"Original box crop: {crop.width} × {crop.height} pixels"
    )

    plt.figure(figsize=(8, 6))
    plt.imshow(crop, interpolation="nearest")
    plt.gca().set_aspect("equal")
    plt.title(f"{row['image_id']} — {row['class']}\n"
              "Original box proportions; no resizing")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

### Additional crop-quality review

Two flagged validation images were inspected using their original
bounding-box regions without resizing:

- **000581 — D10:** A faint horizontal crack-like feature is visible
  in a 95 × 26-pixel region.
- **000997 — D00:** A narrow vertical crack-like feature is visible
  through portions of painted road markings.

Both remain **REVIEW** because their labels cannot be confidently
verified from the available detail. Their crops are retained;
neither case is counted as a confirmed annotation error.

**Limitation:** Direct resizing of narrow regions to 224 × 224
distorts their proportions. Limited source detail also contributes
to uncertainty.

The observations are saved in the quality-review CSV and
`reports/m1/additional_quality_review.md`. This targeted review is
separate from the random label audit.

In [ ]:
from pathlib import Path
from datetime import datetime
import shutil
import pandas as pd

review_path = (
    PROJECT_ROOT / "data" / "processed"
    / "other_quality_visual_review.csv"
)
report_dir = PROJECT_ROOT / "reports" / "m1"
report_dir.mkdir(parents=True, exist_ok=True)

review_df = pd.read_csv(review_path)

observations = {
    "China_Drone_China_Drone_000581": (
        "Original D10 bounding-box region inspected without resizing "
        "(95 x 26 pixels). Faint horizontal crack-like feature visible; "
        "limited source detail prevents confident label verification. "
        "Retained with uncertainty."
    ),
    "China_Drone_China_Drone_000997": (
        "Original D00 bounding-box region inspected without resizing. "
        "Narrow vertical crack-like feature visible through portions "
        "of painted markings; label cannot be confidently verified. "
        "Retained with uncertainty."
    ),
}

# Validate matches before changing or saving anything.
for image_id in observations:
    count = review_df["image_id"].eq(image_id).sum()
    if count != 1:
        raise ValueError(
            f"Expected one review row for {image_id}; found {count}."
        )

# Preserve the existing review file.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
backup_path = review_path.with_name(
    f"{review_path.stem}_backup_{timestamp}.csv"
)
shutil.copy2(review_path, backup_path)

# Use a separate column so earlier notes are preserved.
note_column = "original_crop_review_notes"
if note_column not in review_df.columns:
    review_df[note_column] = ""
review_df[note_column] = review_df[note_column].fillna("").astype(str)

for image_id, note in observations.items():
    mask = review_df["image_id"].eq(image_id)
    review_df.loc[mask, "review_decision"] = "REVIEW"
    review_df.loc[mask, note_column] = note

review_df.to_csv(review_path, index=False, encoding="utf-8")

report = """# Additional image-quality review

## Scope
Two flagged China Drone images were inspected using their original
bounding-box regions without square resizing. Both contribute one
crop to the validation set.

## Findings
- **China_Drone_China_Drone_000581 (D10):** The original box region is
  95 × 26 pixels. A faint horizontal crack-like feature is visible,
  but limited detail prevents confident label verification.
- **China_Drone_China_Drone_000997 (D00):** A narrow vertical crack-like
  feature is visible through portions of painted markings, but the
  label cannot be confidently verified.

## Decision
Both records remain REVIEW and their crops are retained.
These are uncertain cases, not confirmed annotation errors.
No images, crops, labels, or split assignments were changed.

## Dataset limitation
Some source annotations cover narrow or low-detail regions. Resizing
these regions directly to 224 × 224 distorts their proportions and
can make visual interpretation harder. Inspection without resizing
does not resolve all label uncertainty.

## Relationship to the label audit
This targeted quality review is separate from the random label audit.
Its findings must not be added to the random audit counts unless the
images are verified members of that audit sample.
"""

report_path = report_dir / "additional_quality_review.md"
report_path.write_text(report, encoding="utf-8")

display(
    review_df.loc[
        review_df["image_id"].isin(observations),
        ["image_id", "review_decision", note_column],
    ]
)

print("Updated CSV:", review_path)
print("Backup:", backup_path)
print("Saved Markdown report:", report_path)